Modified and verified by **Heejoon Moon**
- version: 11.28

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
print(os.listdir('/content/drive/MyDrive'))


['Ã¡â€žÂÃ¡â€¦Â¥Ã¡â€ Â·Ã¡â€žâ€˜Ã¡â€¦Â²Ã¡â€žÂÃ¡â€¦Â¥Ã¡â€žâ€¡Ã¡â€¦ÂµÃ¡â€žÅ’Ã¡â€¦Â¥Ã¡â€ Â«Ã¡â€žâ‚¬Ã¡â€¦Â¢Ã¡â€žâ€¦Ã¡â€¦Â©Ã¡â€ Â«', 'Colab Notebooks', 'Toa_test_sort.csv', 'Toa_sort.csv', '.ipynb_checkpoints', 'npy_dataset']


In [3]:
source_path = '/content/drive/MyDrive/Ã¬Â»Â´Ã­â€œÂ¨Ã­â€žÂ°Ã«Â¹â€žÃ¬Â â€žÃªÂ°Å“Ã«Â¡Â '  # Ã¬â€¹Â¤Ã¬Â Å“ Ã¬ÂÂ´Ã«Â¦â€ž Ã«Â§Å¾Ã¬Â¶Â°Ã¬â€žÅ“
print(os.listdir(source_path))          # ['train', 'val'] Ã«â€šËœÃ¬â„¢â‚¬Ã¬â€¢Â¼ Ã­â€¢Â¨
print(os.listdir(source_path + '/train'))  # ['Fight', 'NonFight'] Ã«â€œÂ±


['train_v3.ipynb', 'calib_image.png', 'cv-harris_corner_skeleton_code.ipynb', 'slow.mp4', 'val', 'test', 'train', '.ipynb_checkpoints', 'models', 'Abuse018_x264.mp4', 'train_v3 (1).ipynb', 'visualization.ipynb', 'output.mp4', 'optical_flow.ipynb']
['Fight', 'NonFight']


## **0. Preprocessing Data**
- concat RGB frames and opticalflow
- transform videos to '.npy' format
  - '.npy' has 5 channels -> RGB (3) and opticalflow ((u, v) -> 2)
  - you can use another opticalflow algorithms

In [18]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
from tqdm import tqdm

def getOpticalFlow(video):
    """Calculate dense optical flow of input video
    Args:
        video: the input video with shape of [frames,height,width,channel]. dtype=np.array
    Returns:
        flows_x: the optical flow at x-axis, with the shape of [frames,height,width,channel]
        flows_y: the optical flow at y-axis, with the shape of [frames,height,width,channel]
    """
    gray_video = []
    for i in range(len(video)):
        img = cv2.cvtColor(video[i], cv2.COLOR_RGB2GRAY)
        gray_video.append(np.reshape(img, (224, 224, 1)))

    flows = []
    for i in range(0, len(video) - 1):
        flow = cv2.calcOpticalFlowFarneback(gray_video[i], gray_video[i + 1], None, 0.5, 3, 15, 3, 5, 1.2,
                                            cv2.OPTFLOW_FARNEBACK_GAUSSIAN)
        flow[..., 0] -= np.mean(flow[..., 0])
        flow[..., 1] -= np.mean(flow[..., 1])
        flow[..., 0] = cv2.normalize(flow[..., 0], None, 0, 255, cv2.NORM_MINMAX)
        flow[..., 1] = cv2.normalize(flow[..., 1], None, 0, 255, cv2.NORM_MINMAX)
        flows.append(flow)

    flows.append(np.zeros((224, 224, 2)))

    return np.array(flows, dtype=np.float32)


def Video2Npy(file_path, resize=(224,224)):
    """Load video and tansfer it into .npy format
    Args:
        file_path: the path of video file
        resize: the target resolution of output video
    Returns:
        frames: gray-scale video
        flows: magnitude video of optical flows
    """
    cap = cv2.VideoCapture(file_path)
    len_frames = int(cap.get(7))
    try:
        frames = []
        for i in range(len_frames-1):
            _, frame = cap.read()
            frame = cv2.resize(frame,resize, interpolation=cv2.INTER_AREA)
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = np.reshape(frame, (224,224,3))
            frames.append(frame)
    except:
        print("Error: ", file_path, len_frames,i)
    finally:
        frames = np.array(frames)
        cap.release()

    flows = getOpticalFlow(frames)

    optical_flow_map = farneback_visual(flows)

    result = np.zeros((len(flows),224,224,5))
    result[...,:3] = frames
    result[...,3:] = flows

    return result

def farneback_visual(flows):
    pass


def Save2Npy(file_dir, save_dir):
    """Transfer all the videos and save them into specified directory
    Args:
        file_dir: source folder of target videos
        save_dir: destination folder of output .npy files
    """
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
    videos = os.listdir(file_dir)
    for v in tqdm(videos):
        video_name = v.split('.')[0]
        video_path = os.path.join(file_dir, v)
        save_path = os.path.join(save_dir, video_name+'.npy')
        data = Video2Npy(file_path=video_path, resize=(224,224))
        data = np.uint8(data)
        np.save(save_path, data)

    return None

### convert data and save it (offline)

In [19]:
source_path = '/content/drive/MyDrive/Ã¬Â»Â´Ã­â€œÂ¨Ã­â€žÂ°Ã«Â¹â€žÃ¬Â â€žÃªÂ°Å“Ã«Â¡Â '
target_path = '/content/drive/MyDrive/Ã¬Â»Â´Ã­â€œÂ¨Ã­â€žÂ°Ã«Â¹â€žÃ¬Â â€žÃªÂ°Å“Ã«Â¡Â '


for f1 in ['train', 'val']:
    for f2 in ['Fight', 'NonFight']:
        path1 = os.path.join(source_path, f1, f2)
        path2 = os.path.join(target_path, f1, f2)
        Save2Npy(file_dir=path1, save_dir=path2)

  0%|          | 0/160 [00:00<?, ?it/s]

Error:  /content/drive/MyDrive/Ã¬Â»Â´Ã­â€œÂ¨Ã­â€žÂ°Ã«Â¹â€žÃ¬Â â€žÃªÂ°Å“Ã«Â¡Â /train/Fight/1O4KGHbRt3M_3.avi 150 72


  1%|          | 1/160 [00:17<46:25, 17.52s/it]


KeyboardInterrupt: 

## **1. Build Data Loader**

In [ ]:
import torch
import torch.utils.data as data
from torch.utils.data import DataLoader, Dataset
import numpy as np
import os
import cv2

class DataGenerator(Dataset):

    def __init__(self, directory, data_augmentation=True, phase='train'):
        self.phase=phase
        self.directory = directory
        self.data_aug = data_augmentation
        self.X_path, self.Y_dict = self.search_data()
        self.print_stats()

    def __len__(self):
        steps_per_epoch = int(len(self.X_path))
        return steps_per_epoch

    def __getitem__(self, index):
        data, label = self.data_generation(self.X_path[index])
        return data.float(), label


    def load_data(self, path):
        data = np.load(path, mmap_mode='r') # Read the raw data from path
        data = self.uniform_sampling(data, target_frames=64) # Randomly sample number of target frames
        if self.data_aug: # If data is augmented...
            data[..., :3] = self.color_jitter(data[..., :3])
            data = self.random_flip(data, prob=0.5) # Random flip image into random direction
        data[..., :3] = self.normalize(data[..., :3]) # Normalize RGB
        data[..., 3:] = self.normalize(data[..., 3:]) # Normalize optical flows
        return data

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

    def normalize(self, data):
        mean = data.mean()
        std = data.std()
        return (data - mean) / std

    def random_flip(self, video, prob):
        s = np.random.rand()
        if s < prob:
            video = np.flip(video, (2,)) # Flip in width direction
        return video

    def uniform_sampling(self, video, target_frames=64):
        len_frames = int(len(video))
        interval = int(np.ceil(len_frames/target_frames))
        sampled_video = []
        for i in range(0,len_frames,interval):
            sampled_video.append(video[i])
        num_pad = target_frames - len(sampled_video) # num pad = # target frame - # current frame
        padding = []
        if num_pad>0:
            for i in range(-num_pad,0):
                try:
                    padding.append(video[i]) # Fill with the last video frame
                except:
                    padding.append(video[0])
            sampled_video += padding # Add padding results
        return np.array(sampled_video, dtype=np.float32)

    def color_jitter(self, video):
        s_jitter = np.random.uniform(-0.2, 0.2)
        v_jitter = np.random.uniform(-30, 30)
        for i in range(len(video)):
            hsv = cv2.cvtColor(np.array(video[i]), cv2.COLOR_RGB2HSV) # Convert RGB -> HSV
            s = hsv[..., 1] + s_jitter # saturation jitter
            v = hsv[..., 2] + v_jitter # Value jitter
            s[s < 0] = 0
            s[s > 1] = 1
            v[v < 0] = 0
            v[v > 255] = 255
            hsv[..., 1] = s # set jittered saturation
            hsv[..., 2] = v # Set jittered value
            video[i] = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB) # Convert HSV -> RGB again
        return video

    def print_stats(self):
            self.n_files = len(self.X_path)
            self.n_classes = len(self.dirs)
            self.indexes = np.arange(len(self.X_path))
            np.random.shuffle(self.indexes)
            print("Found {} files belonging to {} classes.".format(self.n_files, self.n_classes))
            for i, label in enumerate(self.dirs):
                print('{:10s} : {}'.format(label, i))


    def search_data(self):
        X_path = []
        Y_dict = {}

        self.dirs = sorted(os.listdir(self.directory))  # Get sorted file directories
        one_hots = np.eye(len(self.dirs), dtype=np.float32)   # ###Fill here###

        for i, folder in enumerate(self.dirs):
          folder_path = os.path.join(self.directory, folder)  # folder_path = .../train/Fight or .../train/NonFight
          for file in os.listdir(folder_path):
                if not file.endswith('.npy'):
                    continue
                file_path = os.path.join(folder_path, file)

                X_path.append(file_path)                        # ###Fill here###

                Y_dict[file_path] = one_hots[i]                 # ###Fill here###

        return X_path, Y_dict

    def data_generation(self, batch_path):
        video = self.load_data(batch_path)       # numpy array (T,H,W,5)
        label = self.Y_dict[batch_path]         # numpy array (num_classes,)

        video = np.ascontiguousarray(video)
        label = np.ascontiguousarray(label)

        video = torch.from_numpy(video).float()
        label = torch.from_numpy(label).float()
        return video, label

## **2. Build Simple Model**
- this model is 'Flow Gated Network' proposed in 'RWF2000'
- you can use off-the-shelf architectures such as ResNet, EfficientNet, etc.
- model structure is produced in below image
- Fully-Connected layer is little bit different with image, so we provide Fully-Connected layer structure

![image.png](attachment:image.png)

In [ ]:
'''
import torch
import torch.nn as nn
import torch.nn.functional as F

class FusionModel(nn.Module):
    def __init__(self):
        super(FusionModel, self).__init__()
        self.relu=nn.ReLU(inplace=True)

        self.rgb_block = nn.Sequential(
            nn.Conv3d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm3d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=(1, 2, 2)),     # ÃªÂ³ÂµÃªÂ°â€žÃ«Â§Å’ downsample

            nn.Conv3d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm3d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=(2, 2, 2)),     # Ã¬â€¹Å“ÃªÂ³ÂµÃªÂ°â€ž downsample
        )
        self.opt_block = nn.Sequential(
            nn.Conv3d(2, 32, kernel_size=3, padding=1),
            nn.BatchNorm3d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=(1, 2, 2)),

            nn.Conv3d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm3d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=(2, 2, 2)),
        )
        self.merge_block = nn.Sequential(
            nn.Conv3d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm3d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool3d((1, 1, 1))   # Ã¢â€ â€™ (B, 128, 1, 1, 1)
        )
        self.fc1 = nn.Linear(128, 128)
        self.dropout = nn.Dropout(0.2)
        self.fc2 = nn.Linear(128, 32)
        self.fc3 = nn.Linear(32, 2)

        self.__init_weight()

    def forward(self, x):
        x = x.transpose(2,4)
        x = x.transpose(3,4)
        x = x.transpose(1,2)
        rgb = x[:,:3,:,:,:]
        opt = x[:,3:5,:,:,:]

        rgb = self.rgb_block(rgb)
        opt = self.opt_block(opt)
        fused = rgb * opt
        fused = F.max_pool3d(fused, kernel_size=2, stride=2)

        merged = self.merge_block(fused)

        x = merged.view(merged.size(0), -1)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)

        return x

    def __init_weight(self):
        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

            elif isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

            elif isinstance(m, nn.BatchNorm3d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
'''

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

class EfficientFusionModel(nn.Module):
    def __init__(self, num_classes=2):
        super(EfficientFusionModel, self).__init__()

        weights = models.EfficientNet_B0_Weights.DEFAULT
        self.rgb_backbone = models.efficientnet_b0(weights=weights)

        self.rgb_backbone.classifier = nn.Identity()
        self.opt_backbone = models.efficientnet_b0(weights=weights)
        self.opt_backbone.classifier = nn.Identity()

        first_conv = self.opt_backbone.features[0][0]
        new_first_conv = nn.Conv2d(
            in_channels=2,
            out_channels=first_conv.out_channels,
            kernel_size=first_conv.kernel_size,
            stride=first_conv.stride,
            padding=first_conv.padding,
            bias=False
        )

        with torch.no_grad():
            new_first_conv.weight[:] = torch.mean(first_conv.weight, dim=1, keepdim=True)

        self.opt_backbone.features[0][0] = new_first_conv

        self.fusion_fc = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(1280 * 2, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        if x.dim() == 5 and x.shape[-1] == 5:
            x = x.permute(0, 4, 1, 2, 3) # (B, 5, T, H, W)

        b, c, t, h, w = x.size()

        rgb = x[:, :3, :, :, :]  # (B, 3, T, H, W)
        opt = x[:, 3:5, :, :, :] # (B, 2, T, H, W)

        rgb = rgb.permute(0, 2, 1, 3, 4).reshape(b * t, 3, h, w)
        opt = opt.permute(0, 2, 1, 3, 4).reshape(b * t, 2, h, w)

        rgb_feat = self.rgb_backbone(rgb)
        opt_feat = self.opt_backbone(opt)

        rgb_feat = rgb_feat.view(b, t, -1)
        opt_feat = opt_feat.view(b, t, -1)

        rgb_feat = torch.mean(rgb_feat, dim=1) # (B, 1280)
        opt_feat = torch.mean(opt_feat, dim=1) # (B, 1280)

        fused = torch.cat((rgb_feat, opt_feat), dim=1) # (B, 2560)

        out = self.fusion_fc(fused)

        return out

## **3. Training the Model**
- set hyper-parameters for training

In [8]:
!pip install wandb

In [20]:
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
from tqdm import tqdm
import wandb


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = EfficientFusionModel().to(device)                          # Ã¢â€ Â model
optimizer = optim.SGD(                                    # Ã¢â€ Âoptimizer
    model.parameters(),
    lr=0.003,
    weight_decay=1e-6,
    momentum=0.9,
    nesterov=True,
)
loss_fn = nn.CrossEntropyLoss()                           # Ã¢â€ Â loss func

trainset_path = '/train'
validation_path = '/val'

trainset_path = '/content/drive/MyDrive/Ã¬Â»Â´Ã­â€œÂ¨Ã­â€žÂ°Ã«Â¹â€žÃ¬Â â€žÃªÂ°Å“Ã«Â¡Â /train'
validation_path = '/content/drive/MyDrive/Ã¬Â»Â´Ã­â€œÂ¨Ã­â€žÂ°Ã«Â¹â€žÃ¬Â â€žÃªÂ°Å“Ã«Â¡Â /val'
train_dataset =DataGenerator(trainset_path,  phase='train', data_augmentation=True)
val_dataset =DataGenerator(validation_path, phase='val',   data_augmentation=False)
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)
val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)




min_loss = np.inf


Found 160 files belonging to 2 classes.
Fight      : 0
NonFight   : 1
Found 40 files belonging to 2 classes.
Fight      : 0
NonFight   : 1


In [21]:
def _train(self):
    model.train()
    acc_temp = 0
    running_loss = 0


    train_acc = acc_temp / len(train_loader.dataset)
    train_loss = running_loss / len(train_loader.dataset)

    return train_acc, train_loss

In [22]:
def _val(self):
    model.eval()
    with torch.no_grad():
        running_loss_val = 0
        acc_temp_val =0


        val_acc = acc_temp_val / len(val_loader.dataset)
        val_loss = running_loss_val / len(val_loader.dataset)

    return val_acc, val_loss

In [25]:
import random
import numpy as np
import os
import torch
import wandb

from torch.cuda.amp import autocast, GradScaler

seed = 0
random.seed(seed)
np.random.seed(seed)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(seed)
if device == 'cuda':
    print('gpu device is using')
    torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
num_epochs = 30

min_loss = float('inf')

scaler = GradScaler()

wandb.login()
wandb.init(project='computer vision', name='Ã¬Â»Â´Ã­â€œÂ¨Ã­â€žÂ°Ã«Â¹â€žÃ¬Â â€žÃªÂ°Å“Ã«Â¡Â ')

for epoch in range(num_epochs):
    print(f"Epoch [{epoch+1}/{num_epochs}] Ã¬â€¹Å“Ã¬Å¾â€˜!")

    train_loss = 0.0
    val_loss = 0.0
    train_correct = 0
    train_total = 0
    val_correct = 0
    val_total = 0


    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()

        with autocast():
            pred = model(x)
            loss = loss_fn(pred, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()

        if y.dim() > 1 and y.size(1) > 1:
            y_labels = torch.argmax(y, dim=1)
        else:
            y_labels = y

        _, predicted = torch.max(pred.data, 1)
        train_total += y_labels.size(0)
        train_correct += (predicted == y_labels).sum().item()

    model.eval()
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            pred = model(x)
            loss = loss_fn(pred, y)
            val_loss += loss.item()
            if y.dim() > 1 and y.size(1) > 1:
                y_labels = torch.argmax(y, dim=1)
            else:
                y_labels = y

            _, predicted = torch.max(pred.data, 1)
            val_total += y_labels.size(0)
            val_correct += (predicted == y_labels).sum().item()

    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)

    avg_train_loss = train_loss / len(train_loader)
    avg_train_acc = 100 * train_correct / train_total

    avg_val_loss = val_loss / len(val_loader)
    avg_val_acc = 100 * val_correct / val_total

    print(f"Epoch [{epoch+1}] Train Acc: {avg_train_acc:.2f}% | Val Acc: {avg_val_acc:.2f}%")
    wandb.log({
        "train_loss": avg_train_loss,
        "train_acc": avg_train_acc,
        "val_loss": avg_val_loss,
        "val_acc": avg_val_acc,
        "epoch": epoch + 1
    })

    save_dir = '/content/drive/MyDrive/Ã¬Â»Â´Ã­â€œÂ¨Ã­â€žÂ°Ã«Â¹â€žÃ¬Â â€žÃªÂ°Å“Ã«Â¡Â /models'
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)

    if avg_val_loss < min_loss:
        min_loss = avg_val_loss
        save_path = os.path.join(save_dir, f'FusionModel_best.pth')
        torch.save(model.state_dict(), save_path)
        print(f"Best model updated at epoch {epoch+1} (Loss: {min_loss:.4f})")

/tmp/ipython-input-1651439947.py:26: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


gpu device is using


Epoch [1/30] Ã¬â€¹Å“Ã¬Å¾â€˜!


/tmp/ipython-input-1651439947.py:50: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch [1] Train Acc: 85.62% | Val Acc: 72.50%
Best model updated at epoch 1 (Loss: 0.4920)
Epoch [2/30] Ã¬â€¹Å“Ã¬Å¾â€˜!
Epoch [2] Train Acc: 85.62% | Val Acc: 72.50%
Epoch [3/30] Ã¬â€¹Å“Ã¬Å¾â€˜!
Epoch [3] Train Acc: 93.75% | Val Acc: 75.00%
Epoch [4/30] Ã¬â€¹Å“Ã¬Å¾â€˜!
Epoch [4] Train Acc: 88.75% | Val Acc: 77.50%
Best model updated at epoch 4 (Loss: 0.4217)
Epoch [5/30] Ã¬â€¹Å“Ã¬Å¾â€˜!
Epoch [5] Train Acc: 80.00% | Val Acc: 85.00%
Epoch [6/30] Ã¬â€¹Å“Ã¬Å¾â€˜!
Epoch [6] Train Acc: 89.38% | Val Acc: 85.00%
Epoch [7/30] Ã¬â€¹Å“Ã¬Å¾â€˜!
Epoch [7] Train Acc: 91.25% | Val Acc: 80.00%
Epoch [8/30] Ã¬â€¹Å“Ã¬Å¾â€˜!
Epoch [8] Train Acc: 92.50% | Val Acc: 75.00%
Epoch [9/30] Ã¬â€¹Å“Ã¬Å¾â€˜!
Epoch [9] Train Acc: 86.88% | Val Acc: 72.50%
Epoch [10/30] Ã¬â€¹Å“Ã¬Å¾â€˜!
Epoch [10] Train Acc: 84.38% | Val Acc: 80.00%
Epoch [11/30] Ã¬â€¹Å“Ã¬Å¾â€˜!
Epoch [11] Train Acc: 88.12% | Val Acc: 72.50%
Epoch [12/30] Ã¬â€¹Å“Ã¬Å¾â€˜!
Epoch [12] Train Acc: 85.00% | Val Acc: 85.00%
Epoch [13/30] Ã¬â€¹Å“Ã¬Å¾â€˜!
Epo

In [26]:
from torch.utils.data import DataLoader




best_model = '/content/drive/MyDrive/Ã¬Â»Â´Ã­â€œÂ¨Ã­â€žÂ°Ã«Â¹â€žÃ¬Â â€žÃªÂ°Å“Ã«Â¡Â /models/FusionModel_best.pth'
test_path = '/content/drive/MyDrive/Ã¬Â»Â´Ã­â€œÂ¨Ã­â€žÂ°Ã«Â¹â€žÃ¬Â â€žÃªÂ°Å“Ã«Â¡Â /val'

test_dataset = DataGenerator(directory=test_path, data_augmentation=False)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=True, num_workers=0)

Found 40 files belonging to 2 classes.
Fight      : 0
NonFight   : 1


In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt

model.load_state_dict(torch.load(best_model, map_location=device))
model.eval()

y_true = []
y_score = []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        pred = model(x)
        score = torch.softmax(pred, dim=1)[:,1].cpu().numpy()
        y_score.extend(score)
        y_true.extend(y.numpy())

pred_labels = (np.array(y_score) > 0.5).astype(int)
true_labels = np.argmax(np.array(y_true), axis=1)  # One-Hot -> Index
accuracy = (pred_labels == true_labels).mean()

print(f'Accuracy: {accuracy:.4f}')

print(f'Accuracy: {accuracy:.4f}')

y_true_index = np.argmax(np.array(y_true), axis=1)

auc = roc_auc_score(y_true_index, y_score)
print(f'AUROC: {auc:.4f}')

fpr, tpr, _ = roc_curve(y_true_index, y_score)
print(f'AUROC: {auc:.4f}')

fpr, tpr, _ = roc_curve(y_true_index, y_score)
plt.plot(fpr, tpr, label=f"AUC={auc:.3f}")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()
